<a href="https://colab.research.google.com/github/kadedamola40-svg/Learning-Journal/blob/main/Hackathon1_Adedamola.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from pathlib import Path
import logging, sys, re
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

RAW_DIR     = Path('data/raw')
PROC_DIR    = Path('data/processed')
LOG_DIR     = Path('logs')
REPORTS_DIR = Path('reports')
FIGURES_DIR = REPORTS_DIR / 'figures'

CSV_FILE      = RAW_DIR / 'orders.csv'
LOG_FILE_PATH = RAW_DIR / 'access.txt'
LOG_OUT       = LOG_DIR / 'hackathon.log'

for p in [RAW_DIR, PROC_DIR, LOG_DIR, REPORTS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)


In [26]:
logger = logging.getLogger('hackathon')
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')

fh = logging.FileHandler(LOG_OUT, encoding='utf-8')
fh.setFormatter(fmt); logger.addHandler(fh)

ch = logging.StreamHandler(sys.stdout)
ch.setFormatter(fmt); logger.addHandler(ch)

logger.info('Setup complete')


2026-05-18 13:25:21,634 | INFO | Setup complete


INFO:hackathon:Setup complete


In [27]:
peek = pd.read_csv(CSV_FILE, nrows=5)
print(peek)
print(peek.columns.tolist())




   order_id  user_id  product_id  quantity             order_ts  \
0     61763      456    80000347         3  2025-08-09 09:57:51   
1     81295     2200    80000041         1  2026-01-07 19:55:11   
2     80453     2665    80000323         3  2026-04-20 18:07:38   
3     71844     1755    80000227         3  2025-10-10 06:43:41   
4     65678     3510    80000097         2  2025-10-30 11:53:42   

             user_name         product_name  unit_price  
0      Suzanne Spencer     DodgerBlue Rerum       32.09  
1        Samantha Holt       DimGray Maxime       31.52  
2  Ashley Gill-Goodwin  PowderBlue Sapiente       34.01  
3     Leanne Middleton  PowderBlue Incidunt       47.52  
4  Mrs Lorraine Arnold            Silver Id       13.03  
['order_id', 'user_id', 'product_id', 'quantity', 'order_ts', 'user_name', 'product_name', 'unit_price']


In [30]:
with open(LOG_FILE_PATH, 'r', encoding='utf-8', errors='ignore') as f: n_lines = sum(1 for _ in f)
print(f'access.log has {n_lines:,} lines')


access.log has 100,695 lines


In [35]:
def load_csv(path: Path) -> pd.DataFrame:
    """Read a CSV and log shape + nulls."""
    logger.info(f'Loading CSV: {path}')
    df = pd.read_csv(path)
    logger.info(f'  rows: {df.shape[0]:,}, cols: {df.shape[1]}')
    null_pct = (df.isna().mean() * 100).round(2)
    logger.info(f'  null %: {null_pct.to_dict()}')
    return df

df_csv = load_csv(CSV_FILE)
df_csv.head()


2026-05-18 13:42:46,456 | INFO | Loading CSV: data/raw/orders.csv


INFO:hackathon:Loading CSV: data/raw/orders.csv


2026-05-18 13:42:46,583 | INFO |   rows: 50,750, cols: 8


INFO:hackathon:  rows: 50,750, cols: 8


2026-05-18 13:42:46,603 | INFO |   null %: {'order_id': 0.0, 'user_id': 0.0, 'product_id': 0.0, 'quantity': 0.0, 'order_ts': 0.0, 'user_name': 2.0, 'product_name': 1.49, 'unit_price': 1.0}


INFO:hackathon:  null %: {'order_id': 0.0, 'user_id': 0.0, 'product_id': 0.0, 'quantity': 0.0, 'order_ts': 0.0, 'user_name': 2.0, 'product_name': 1.49, 'unit_price': 1.0}


,order_id,user_id,product_id,quantity,order_ts,user_name,product_name,unit_price
0,61763,456,80000347,3,2025-08-09 09:57:51,Suzanne Spencer,DodgerBlue Rerum,32.09
1,81295,2200,80000041,1,2026-01-07 19:55:11,Samantha Holt,DimGray Maxime,31.52
2,80453,2665,80000323,3,2026-04-20 18:07:38,Ashley Gill-Goodwin,PowderBlue Sapiente,34.01
3,71844,1755,80000227,3,2025-10-10 06:43:41,Leanne Middleton,PowderBlue Incidunt,47.52
4,65678,3510,80000097,2,2025-10-30 11:53:42,Mrs Lorraine Arnold,Silver Id,13.03


In [50]:
def parse_log_line(line):
    try:
        ip = line.split(' ')[0]

        ts = line[ line.index('[')+1 : line.index(']') ]

        quoted = line.split('"')
        method, path, *_ = quoted[1].split(' ')
        after  = quoted[2].strip().split()
        status = int(after[0])
        size   = int(after[1]) if after[1].isdigit() else None

        return {'ip': ip, 'timestamp': ts,
                'method': method, 'path': path,
                'status': status, 'size': size}
    except (ValueError, IndexError):
        return None